# DR9 Sweep Footprint Versus redMaPPer Cluster Centers

This notebook diagnoses whether redMaPPer cluster centers lie inside the DR9 sweep footprint. It makes Mollweide/`healpy.mollview` maps of the DR9 sweep coverage and overlays the BCG/central cluster positions from `RA_x`, `DEC_x`.

It also highlights clusters that have zero DR9 galaxy counts or no touching sweep file in the local-overdensity output. This is meant to diagnose edge/pathology cases such as the \(165/\sim6000\) clusters with no galaxy counts.

In [ ]:
from pathlib import Path
from glob import glob
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table, unique
from astropy.units import UnitsWarning

import healpy as hp

warnings.filterwarnings(
    'ignore',
    message=r'.*did not parse as fits unit.*',
    category=UnitsWarning,
)

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == 'local_overdensity':
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

LOCAL_OVERDENSITY_DIR = REPO_ROOT / 'local_overdensity'
CATALOG_DIR = REPO_ROOT / 'catalogs'
OUTPUT_DIR = LOCAL_OVERDENSITY_DIR / 'dr9_outputs'
RADIUS_GRID_DIR = OUTPUT_DIR / 'radius_grid_overdensity'
PLOT_DIR = OUTPUT_DIR / 'sweep_footprint_diagnostics'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Plot dir:', PLOT_DIR)

In [ ]:
# Configuration
CLUSTER_CATALOG = CATALOG_DIR / 'rm_clusters_with_spec_richness.pickle'
LOCAL_OVERDENSITY_TABLE = RADIUS_GRID_DIR / 'rm_dr9_radius_grid_overdensity.fits'

SWEEP_DIRS = [
    Path('/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/north/sweep/9.0'),
    Path('/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/south/sweep/9.0'),
]
SWEEP_PATTERN = 'sweep-*.fits'
MAX_SWEEP_FILES = None  # set to e.g. 20 for a quick test

NSIDE_FOOTPRINT = 256
NSIDE_DENSITY = 128

# Which radius definition to use when selecting zero-count clusters from the MPI output.
# None means use the first radius_def_id found in the table.
DIAG_RADIUS_DEF_ID = None

print('Cluster catalog:', CLUSTER_CATALOG)
print('Local overdensity table:', LOCAL_OVERDENSITY_TABLE)
print('Sweep dirs:')
for d in SWEEP_DIRS:
    print(' ', d)

## Helper Functions

In [ ]:
def read_pickle_table(path):
    with Path(path).open('rb') as handle:
        obj = pickle.load(handle)
    if isinstance(obj, Table):
        return obj
    if hasattr(obj, 'to_pandas'):
        return Table.from_pandas(obj.to_pandas())
    return Table(obj)


def col_float(table, col):
    arr = np.ma.asarray(table[col], dtype=float)
    return np.ma.filled(arr, np.nan)


def find_sweep_files(sweep_dirs, pattern='sweep-*.fits', max_files=None):
    files = []
    for sweep_dir in sweep_dirs:
        sweep_dir = Path(sweep_dir)
        files.extend(glob(str(sweep_dir / pattern)))
        files.extend(glob(str(sweep_dir / '*' / pattern)))
    files = sorted(set(files))
    if max_files is not None:
        files = files[:max_files]
    if len(files) == 0:
        raise FileNotFoundError('No sweep files found. Check SWEEP_DIRS and SWEEP_PATTERN.')
    return [Path(f) for f in files]


def _parse_signed_dec(text):
    sign = -1 if text[0].lower() == 'm' else 1
    return sign * float(text[1:])


def parse_sweep_bounds_from_name(path):
    """Parse Legacy Survey sweep bounds from names like sweep-000m005-010p000.fits."""
    name = Path(path).name
    match = re.match(r'sweep-(\d{3})([pm]\d{3})-(\d{3})([pm]\d{3})\.fits(?:\.fz)?$', name)
    if match is None:
        return None
    ra_min = float(match.group(1))
    dec_min = _parse_signed_dec(match.group(2))
    ra_max = float(match.group(3))
    dec_max = _parse_signed_dec(match.group(4))
    return dict(ra_min=ra_min, ra_max=ra_max, dec_min=dec_min, dec_max=dec_max, filename=str(path))


def read_sweep_bounds_from_file(path):
    tab = Table.read(path, hdu=1, memmap=True)
    ra = np.asarray(tab['RA'], dtype=float)
    dec = np.asarray(tab['DEC'], dtype=float)
    good = np.isfinite(ra) & np.isfinite(dec)
    return dict(
        ra_min=float(np.nanmin(ra[good])),
        ra_max=float(np.nanmax(ra[good])),
        dec_min=float(np.nanmin(dec[good])),
        dec_max=float(np.nanmax(dec[good])),
        filename=str(path),
    )


def get_sweep_bounds(files, use_filename=True):
    rows = []
    for i, path in enumerate(files, start=1):
        if i % 100 == 0 or i == len(files):
            print(f'Bounds {i:,}/{len(files):,}')
        bounds = parse_sweep_bounds_from_name(path) if use_filename else None
        if bounds is None:
            bounds = read_sweep_bounds_from_file(path)
        rows.append(bounds)
    return Table(rows=rows)


def build_box_footprint_map(bounds_table, nside=256):
    """Mark HEALPix pixels whose centers fall inside any sweep RA/DEC box."""
    npix = hp.nside2npix(nside)
    theta, phi = hp.pix2ang(nside, np.arange(npix))
    ra_pix = np.rad2deg(phi)
    dec_pix = 90.0 - np.rad2deg(theta)
    footprint = np.zeros(npix, dtype=np.int16)

    for i, row in enumerate(bounds_table):
        if i % 100 == 0 or i == len(bounds_table) - 1:
            print(f'Footprint box {i + 1:,}/{len(bounds_table):,}')
        ra_min = float(row['ra_min'])
        ra_max = float(row['ra_max'])
        dec_min = float(row['dec_min'])
        dec_max = float(row['dec_max'])
        dec_mask = (dec_pix >= dec_min) & (dec_pix < dec_max)
        if ra_min <= ra_max:
            ra_mask = (ra_pix >= ra_min) & (ra_pix < ra_max)
        else:
            ra_mask = (ra_pix >= ra_min) | (ra_pix < ra_max)
        footprint[ra_mask & dec_mask] = 1
    return footprint


def point_density_map(ra, dec, nside=128):
    ra = np.asarray(ra, dtype=float)
    dec = np.asarray(dec, dtype=float)
    good = np.isfinite(ra) & np.isfinite(dec)
    pix = hp.ang2pix(nside, np.deg2rad(90.0 - dec[good]), np.deg2rad(ra[good]))
    counts = np.zeros(hp.nside2npix(nside), dtype=float)
    np.add.at(counts, pix, 1)
    return counts / hp.nside2pixarea(nside, degrees=True)


def in_any_sweep_box(ra, dec, bounds_table):
    ra = np.asarray(ra, dtype=float)
    dec = np.asarray(dec, dtype=float)
    inside = np.zeros(len(ra), dtype=bool)
    finite = np.isfinite(ra) & np.isfinite(dec)
    for row in bounds_table:
        ra_min = float(row['ra_min'])
        ra_max = float(row['ra_max'])
        dec_min = float(row['dec_min'])
        dec_max = float(row['dec_max'])
        dec_mask = (dec >= dec_min) & (dec < dec_max)
        if ra_min <= ra_max:
            ra_mask = (ra >= ra_min) & (ra < ra_max)
        else:
            ra_mask = (ra >= ra_min) | (ra < ra_max)
        inside |= finite & ra_mask & dec_mask
    return inside

## Load Sweep Files And Cluster Centers

In [ ]:
sweep_files = find_sweep_files(SWEEP_DIRS, SWEEP_PATTERN, max_files=MAX_SWEEP_FILES)
print(f'Found {len(sweep_files):,} sweep files')

sweep_bounds = get_sweep_bounds(sweep_files, use_filename=True)
print(sweep_bounds[:5])

cluster_table = read_pickle_table(CLUSTER_CATALOG)
if 'ID' in cluster_table.colnames:
    cluster_table = unique(cluster_table, keys='ID')

ra_rm = col_float(cluster_table, 'RA_x')
dec_rm = col_float(cluster_table, 'DEC_x')
z_rm = col_float(cluster_table, 'Z_SPEC_x') if 'Z_SPEC_x' in cluster_table.colnames else col_float(cluster_table, 'Z_LAMBDA')

inside_sweep_box = in_any_sweep_box(ra_rm, dec_rm, sweep_bounds)
print(f'Clusters: {len(cluster_table):,}')
print(f'Cluster centers inside at least one sweep box: {np.count_nonzero(inside_sweep_box):,}/{len(cluster_table):,}')
print(f'Cluster centers outside all sweep boxes: {np.count_nonzero(~inside_sweep_box):,}')

## Load Local-Overdensity Output And Define Problem Clusters

The MPI output is used to identify clusters with zero counts or no touching sweep file. If the table contains multiple radius definitions, this cell uses `DIAG_RADIUS_DEF_ID` or defaults to the first radius definition.

In [ ]:
if LOCAL_OVERDENSITY_TABLE.exists():
    od_table_all = Table.read(LOCAL_OVERDENSITY_TABLE)
    radius_ids = np.unique(np.asarray(od_table_all['radius_def_id'], dtype=int)) if 'radius_def_id' in od_table_all.colnames else np.array([0])
    if DIAG_RADIUS_DEF_ID is None:
        DIAG_RADIUS_DEF_ID = int(radius_ids[0])
    od_table = od_table_all[np.asarray(od_table_all['radius_def_id'], dtype=int) == DIAG_RADIUS_DEF_ID]
    print(f'Loaded local-overdensity rows: {len(od_table_all):,}')
    print(f'Using radius_def_id={DIAG_RADIUS_DEF_ID}; rows={len(od_table):,}')

    n_signal = col_float(od_table, 'N_signal') if 'N_signal' in od_table.colnames else np.full(len(od_table), np.nan)
    n_bg = col_float(od_table, 'N_background') if 'N_background' in od_table.colnames else np.full(len(od_table), np.nan)
    n_touch = col_float(od_table, 'n_files_touching_cluster') if 'n_files_touching_cluster' in od_table.colnames else np.full(len(od_table), np.nan)
    coverage_signal = col_float(od_table, 'coverage_signal') if 'coverage_signal' in od_table.colnames else np.full(len(od_table), np.nan)
    coverage_bg = col_float(od_table, 'coverage_background') if 'coverage_background' in od_table.colnames else np.full(len(od_table), np.nan)
    sigma_excess = col_float(od_table, 'Sigma_excess') if 'Sigma_excess' in od_table.colnames else np.full(len(od_table), np.nan)

    ra_diag = col_float(od_table, 'RA_x')
    dec_diag = col_float(od_table, 'DEC_x')

    no_touch = np.isfinite(n_touch) & (n_touch <= 0)
    zero_signal = np.isfinite(n_signal) & (n_signal == 0)
    zero_bg = np.isfinite(n_bg) & (n_bg == 0)
    zero_both_counts = zero_signal & zero_bg
    zero_excess = np.isfinite(sigma_excess) & (np.abs(sigma_excess) <= 1e-12)
    low_coverage = (np.isfinite(coverage_signal) & (coverage_signal < 0.8)) | (np.isfinite(coverage_bg) & (coverage_bg < 0.8))

    print(f'No touching sweep file: {np.count_nonzero(no_touch):,}')
    print(f'Zero signal count: {np.count_nonzero(zero_signal):,}')
    print(f'Zero background count: {np.count_nonzero(zero_bg):,}')
    print(f'Zero signal and background count: {np.count_nonzero(zero_both_counts):,}')
    print(f'Zero Sigma_excess: {np.count_nonzero(zero_excess):,}')
    print(f'Coverage < 0.8 in signal or background: {np.count_nonzero(low_coverage):,}')
else:
    print('Local-overdensity table not found. Only footprint-vs-cluster-center diagnostics will be made.')
    od_table = None

## Build DR9 Sweep Footprint Map

This map is based on the RA/DEC boxes encoded in the sweep-file names. It is intentionally a footprint diagnostic rather than a density map.

In [ ]:
footprint_map = build_box_footprint_map(sweep_bounds, nside=NSIDE_FOOTPRINT)
footprint_plot = footprint_map.astype(float)
footprint_plot[footprint_map == 0] = hp.UNSEEN

rm_density = point_density_map(ra_rm, dec_rm, nside=NSIDE_DENSITY)
rm_density_plot = rm_density.copy()
rm_density_plot[rm_density <= 0] = hp.UNSEEN

print(f'Footprint pixels covered: {np.count_nonzero(footprint_map > 0):,}/{len(footprint_map):,}')

## Mollview: DR9 Sweep Footprint With All RM Centers

In [ ]:
fig = plt.figure(figsize=(12, 7))
hp.mollview(
    footprint_plot,
    fig=fig.number,
    title='DR9 sweep footprint boxes with redMaPPer BCG centers',
    unit='inside sweep box',
    cmap='Greys',
    min=0,
    max=1,
)
hp.projscatter(ra_rm, dec_rm, lonlat=True, s=3, alpha=0.35, color='royalblue', label='RM centers')
hp.projscatter(ra_rm[~inside_sweep_box], dec_rm[~inside_sweep_box], lonlat=True, s=18, alpha=0.95, color='crimson', label='outside sweep boxes')
hp.graticule(color='white', alpha=0.25)
plt.legend(loc='lower left', bbox_to_anchor=(0.02, 0.02), frameon=False)
fig.savefig(PLOT_DIR / 'dr9_sweep_footprint_rm_centers_mollview.png', dpi=180, bbox_inches='tight')
plt.show()

## Mollview: Highlight Zero-Count / Edge Clusters

In [ ]:
if od_table is not None:
    fig = plt.figure(figsize=(12, 7))
    hp.mollview(
        footprint_plot,
        fig=fig.number,
        title=f'DR9 footprint with zero-count diagnostics, radius_def_id={DIAG_RADIUS_DEF_ID}',
        unit='inside sweep box',
        cmap='Greys',
        min=0,
        max=1,
    )
    hp.projscatter(ra_diag, dec_diag, lonlat=True, s=2, alpha=0.22, color='0.2', label='all clusters')
    hp.projscatter(ra_diag[zero_both_counts], dec_diag[zero_both_counts], lonlat=True, s=15, alpha=0.9, color='darkorange', label='Nsig=Nbg=0')
    hp.projscatter(ra_diag[no_touch], dec_diag[no_touch], lonlat=True, s=28, alpha=0.95, color='crimson', marker='x', label='no touching sweep')
    hp.projscatter(ra_diag[low_coverage], dec_diag[low_coverage], lonlat=True, s=18, alpha=0.8, facecolors='none', edgecolors='limegreen', label='coverage < 0.8')
    hp.graticule(color='white', alpha=0.25)
    plt.legend(loc='lower left', bbox_to_anchor=(0.02, 0.02), frameon=False)
    fig.savefig(PLOT_DIR / 'dr9_sweep_footprint_zero_count_clusters_mollview.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('Skipping zero-count overlay because local-overdensity output was not found.')

## Zoomed RA/DEC Diagnostic

The Mollweide projection is useful globally, but edge failures are often easier to see in ordinary RA/DEC coordinates.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))

# Draw sweep boxes as thin outlines.
for row in sweep_bounds:
    ra_min = float(row['ra_min'])
    ra_max = float(row['ra_max'])
    dec_min = float(row['dec_min'])
    dec_max = float(row['dec_max'])
    if ra_max >= ra_min:
        ax.plot([ra_min, ra_max, ra_max, ra_min, ra_min], [dec_min, dec_min, dec_max, dec_max, dec_min], color='0.8', lw=0.35, alpha=0.5)

ax.scatter(ra_rm, dec_rm, s=5, alpha=0.25, color='royalblue', label='RM centers')
ax.scatter(ra_rm[~inside_sweep_box], dec_rm[~inside_sweep_box], s=24, alpha=0.9, color='crimson', label='outside sweep boxes')
if od_table is not None:
    ax.scatter(ra_diag[zero_both_counts], dec_diag[zero_both_counts], s=18, alpha=0.85, color='darkorange', label='Nsig=Nbg=0')
    ax.scatter(ra_diag[no_touch], dec_diag[no_touch], s=40, alpha=0.95, color='crimson', marker='x', label='no touching sweep')

ax.set_xlabel('RA [deg]')
ax.set_ylabel('Dec [deg]')
ax.set_title('DR9 sweep boxes and redMaPPer cluster centers')
ax.legend(frameon=False, markerscale=1.4)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'dr9_sweep_boxes_rm_centers_radec.png', dpi=180)
plt.show()

## Tables Of Suspect Clusters

These tables are useful for checking whether the affected clusters are all near the same sky boundary or have unusual redshifts/richnesses.

In [ ]:
if od_table is not None:
    keep_cols = [col for col in ['ID', 'RA_x', 'DEC_x', 'Z_SPEC_x', 'Z_LAMBDA', 'LAMBDA', 'lambda_spec_tot', 'radius_def_id', 'radius_label', 'N_signal', 'N_background', 'coverage_signal', 'coverage_background', 'n_files_touching_cluster', 'Sigma_excess'] if col in od_table.colnames]

    no_touch_table = od_table[no_touch][keep_cols]
    zero_both_table = od_table[zero_both_counts][keep_cols]
    low_coverage_table = od_table[low_coverage][keep_cols]

    no_touch_table.write(PLOT_DIR / 'clusters_no_touching_sweep.ecsv', format='ascii.ecsv', overwrite=True)
    zero_both_table.write(PLOT_DIR / 'clusters_zero_signal_background_counts.ecsv', format='ascii.ecsv', overwrite=True)
    low_coverage_table.write(PLOT_DIR / 'clusters_low_sweep_coverage.ecsv', format='ascii.ecsv', overwrite=True)

    print('No touching sweep file:')
    display(no_touch_table[:20])
    print('Zero signal and background counts:')
    display(zero_both_table[:20])
    print('Low coverage:')
    display(low_coverage_table[:20])
    print('Saved diagnostic tables to:', PLOT_DIR)
else:
    outside_table = cluster_table[~inside_sweep_box]
    outside_table.write(PLOT_DIR / 'cluster_centers_outside_sweep_boxes.ecsv', format='ascii.ecsv', overwrite=True)
    display(outside_table[:20])

## Saved Outputs

In [ ]:
print('Saved diagnostic outputs in:', PLOT_DIR)
for path in sorted(PLOT_DIR.glob('*')):
    print(path.name)